In [0]:
dbutils.library.restartPython()

In [0]:
container = "bronze"
adls_location = "vedatalakestoragedemo"
bronze_file_path = "demo_location/taxi_data/bronze"
silver_file_path = "demo_location/taxi_data/silver"

df_trip = spark.read.option("header", "true").option("inferSchema", "true").csv(f"abfss://{container}@{adls_location}.dfs.core.windows.net/{bronze_file_path}/trip_data_1.csv")

df_fare = spark.read.option("header", "true").option("inferSchema", "true").csv(f"abfss://{container}@{adls_location}.dfs.core.windows.net/{bronze_file_path}/trip_fare_1.csv")

trip_name = "taxi_trip_data"
fare_name = "taxi_fare_data"

silver_layer = f"abfss://{container}@{adls_location}.dfs.core.windows.net/{silver_file_path}"


In [0]:
%run
"../utility_functions/utility_function.py"

In [0]:
# List all functions and variables defined in the utility_function script
defined_items = dir()
print(defined_items)

In [0]:
# Utility imports for saving data as Delta tables
from databricks.data_engineering.src.utility_functions.utility_function.py import save_cleaned_data



def clean_dataset(df):
    """
    Remove duplicate rows, strip whitespace from column names, and filter out rows with vendor_id 'CMT'.
    """
    df_no_duplicates = df.dropDuplicates()
    df_stripped_columns = df_no_duplicates.toDF(*(c.strip() for c in df_no_duplicates.columns))
    df_filtered = df_stripped_columns.filter(df_stripped_columns.vendor_id != 'CMT')
    return df_filtered

def dropna_by_column(df, column):
    """
    Drop rows where the specified column has null values.
    """
    return df.dropna(subset=[column])

def prepare_df_fare(df):
    """
    Clean fare DataFrame and drop rows with null 'total_amount'.
    """
    df_clean = clean_dataset(df)
    df_prep = dropna_by_column(df_clean, 'total_amount')
    return df_prep

def prepare_df_trip(df):
    """
    Clean trip DataFrame and drop rows with null 'trip_distance'.
    """
    df_clean = clean_dataset(df)
    df_prep = dropna_by_column(df_clean, 'trip_distance')
    return df_prep


def bronze_to_silver_taxi(df_trip, df_fare, trip_name, fare_name, silver_file_path):
    """
    Clean and prepare trip and fare DataFrames, then save them to the silver layer as Delta tables.
    """
    df_trip_cleaned = prepare_df_trip(df_trip)
    df_fare_cleaned = prepare_df_fare(df_fare)

    save_cleaned_data(df_trip_cleaned, trip_name, silver_file_path)
    save_cleaned_data(df_fare_cleaned, fare_name, silver_file_path)

In [0]:
bronze_to_silver_taxi(
    df_trip=df_trip, 
    df_fare=df_fare, 
    trip_name=trip_name, 
    fare_name=fare_name, 
    silver_file_path=silver_layer)

In [0]:
bronze_to_silver_taxi()

In [0]:
%sql
from azure_cloud.default.taxi_trip_data_cleaned select *

In [0]:
df_trip_cleaned = prepare_df_trip(df_trip)
df_trip_cleaned.printSchema()
df_trip_cleaned.display()



In [0]:
df_fare_cleaned = prepare_df_fare(df_fare)
df_fare_cleaned.printSchema()
df_fare_cleaned.display()